# Probability grids

This notebook creates the time-dependent probability maps and writes them to file (`.nc` format), using the models trained in the previous notebook (`01-create_classifiers.ipynb`). That notebook must have been run previously. Paths follow [`PathConfigManager`](lib/paths.py).

## Notebook setup

These cells set paths and run parameters from the selected config file.

### Config

In [3]:
config_file = "config/.run_config.yml"

In [4]:
from lib.paths import PathConfigManager

pcm = PathConfigManager(config_file, notebook="02")

# =====================
# Filestructure
# =====================

grid_data_filepath = pcm.GRID_DATA_PATH
training_filepath = pcm.TRAINING_DATA_PATH
output_dir = pcm.OUTPUT_DIR

pcm.create_directories()


# =====================
# Notebook scope
# =====================

use_extracted_data = pcm.use_extracted_data

# =====================
# Run parameters
# =====================

n_jobs = pcm.config["n_jobs"]
overwrite = pcm.config["overwrite_output"]
verbose = pcm.config["verbose"]

### Imports

In [5]:
import time
from datetime import timedelta
from pathlib import Path

import pandas as pd
from joblib import load

from lib.check_files import check_prepared_data
from lib.pu import create_probability_grids

objc[92133]: Class QT_ROOT_LEVEL_POOL__THESE_OBJECTS_WILL_BE_RELEASED_WHEN_QAPP_GOES_OUT_OF_SCOPE is implemented in both /Users/glados/opt/anaconda3/envs/prospectivity/lib/libQt6Core.6.11.0.dylib (0x1a0270738) and /Users/glados/opt/anaconda3/envs/prospectivity/lib/libQt5Core.5.15.15.dylib (0x19467b3d0). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[92133]: Class KeyValueObserver is implemented in both /Users/glados/opt/anaconda3/envs/prospectivity/lib/libQt6Core.6.11.0.dylib (0x1a0270760) and /Users/glados/opt/anaconda3/envs/prospectivity/lib/libQt5Core.5.15.15.dylib (0x19467b3f8). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[92133]: Class RunLoopModeTracker is implemented in both /Users/glados/opt/anaconda3/envs/prospectivity/lib/libQt6Core.6.11.0.dylib (0x1a02707b0) and /Users/glados/opt/anaconda3/envs/prospectivity/lib/libQt5Core.5.15.1

### Input and output files

In [ ]:
if use_extracted_data:
    if verbose:
        print(f"Using extracted grid/training data: {grid_data_filepath}, {training_filepath}")
else:
    prep_dir = check_prepared_data(pcm.PREPARED_DATA_DIR, verbose=True)
    prep_dir = Path(prep_dir)
    grid_data_filepath = prep_dir / "grid_data.csv"
    training_filepath = prep_dir / "training_data_global.csv"

columns = [
    "lon",
    "lat",
    "present_lon",
    "present_lat",
    "age (Ma)",
    "erosion (m)",
    "region",
]

point_data = pd.read_csv(grid_data_filepath)
df_out = point_data[columns]
regions = list(pd.read_csv(training_filepath)["region"].unique())

Using extracted grid/training data: /Users/glados/Documents/Not Useless/Documents/University/2026/Honours/Data & Code/PUB-framework-Alfonso/data_extracted/zahirovic2022/polygons_points/trenches_9.0_deg_buffer/grid_data.csv, /Users/glados/Documents/Not Useless/Documents/University/2026/Honours/Data & Code/PUB-framework-Alfonso/data_extracted/zahirovic2022/polygons_points/trenches_9.0_deg_buffer/training_data_global.csv


## Calculate probabilities

Deposit probability will be calculated for the gridded data and written to a CSV file.

In [ ]:
# algorithms = ("PU", "SVM") if create_SVM else ("PU",)
algorithms = ("PU",)
for algorithm in algorithms:
    t0 = time.time()

    subdir = output_dir / algorithm
    model_filename = subdir / "classifier.joblib"
    if not model_filename.is_file():
        continue
    probabilities_filename = subdir / "grid_probabilities.csv"
    model = load(model_filename)

    point_x = point_data[model.feature_names_in_]
    prob_scores = model.predict_proba(point_x)[:, 1].ravel()
    probabilities = df_out.copy()
    probabilities["probability"] = prob_scores
    del prob_scores
    probabilities.to_csv(probabilities_filename, index=False)
    del probabilities, model
    duration = timedelta(seconds=time.time() - t0)
    if verbose:
        print(
            f"Calculating probabilities for {algorithm} model complete",
            f"(region: global; duration: {duration})",
        )

    for region in regions:
        region_code = "_".join(region.lower().split())
        model_filename = subdir / f"classifier_{region_code}.joblib"
        if not model_filename.is_file():
            continue
        t0 = time.time()

        probabilities_filename = subdir / f"grid_probabilities_{region_code}.csv"
        model = load(model_filename)

        try:
            model[-1].set_params(n_jobs=1)
        except ValueError:
            pass

        point_x = point_data[model.feature_names_in_]
        prob_scores = model.predict_proba(point_x)[:, 1].ravel()
        probabilities = df_out.copy()
        probabilities["probability"] = prob_scores
        del prob_scores
        probabilities.to_csv(probabilities_filename, index=False)
        del probabilities, model
        duration = timedelta(seconds=time.time() - t0)
        if verbose:
            print(
                f"Calculating probabilities for {algorithm} model complete",
                f"(region: {region}; duration: {duration})",
            )

del point_data

Calculating probabilities for PU model complete (region: global; duration: 0:01:25.264917)
Calculating probabilities for PU model complete (region: North America; duration: 0:03:44.921019)
Calculating probabilities for PU model complete (region: South America; duration: 0:03:24.212240)
Calculating probabilities for PU model complete (region: Southeast Asia; duration: 0:03:20.212177)
Calculating probabilities for PU model complete (region: Tethys; duration: 0:03:05.222457)
Calculating probabilities for SVM model complete (region: global; duration: 0:02:26.733044)
Calculating probabilities for SVM model complete (region: North America; duration: 0:00:48.735408)
Calculating probabilities for SVM model complete (region: South America; duration: 0:02:49.978118)


## Create probability maps

The probabilities calculated in the previous section will now be written to one netCDF file per time step:

In [ ]:
# algorithms = ("PU", "SVM") if create_SVM else ("PU",)
algorithms = ("PU",)
for algorithm in algorithms:
    t0 = time.time()

    subdir = output_dir / algorithm
    probabilities_filename = subdir / "grid_probabilities.csv"
    if not probabilities_filename.is_file():
        continue
    grid_output_dir = subdir / "probability_grids"
    grid_output_dir.mkdir(parents=True, exist_ok=True)

    create_probability_grids(
        data=str(probabilities_filename),
        output_dir=str(grid_output_dir),
        threads=n_jobs,
        extent=(-180, 180, -90, 90),
    )
    duration = timedelta(seconds=time.time() - t0)
    if verbose:
        print(
            f"Creating grids for {algorithm} model complete",
            f"(region: global; duration: {duration})",
        )

    for region in regions:
        region_code = "_".join(region.lower().split())
        subdir = output_dir / algorithm
        probabilities_filename = subdir / f"grid_probabilities_{region_code}.csv"
        if not probabilities_filename.is_file():
            continue
        t0 = time.time()

        grid_output_dir = subdir / f"probability_grids_{region_code}"
        grid_output_dir.mkdir(parents=True, exist_ok=True)

        create_probability_grids(
            data=str(probabilities_filename),
            output_dir=str(grid_output_dir),
            threads=n_jobs,
            extent=(-180, 180, -90, 90),
        )
        duration = timedelta(seconds=time.time() - t0)
        if verbose:
            print(
                f"Creating grids for {algorithm} model complete",
                f"(region: {region}; duration: {duration})",
            )

objc[51430]: Class QT_ROOT_LEVEL_POOL__THESE_OBJECTS_WILL_BE_RELEASED_WHEN_QAPP_GOES_OUT_OF_SCOPE is implemented in both /Users/glados/opt/anaconda3/envs/pub-alfonso/lib/libQt6Core.6.10.2.dylib (0x1556785b0) and /Users/glados/opt/anaconda3/envs/pub-alfonso/lib/libQt5Core.5.15.15.dylib (0x1495c73d0). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[51429]: Class QT_ROOT_LEVEL_POOL__THESE_OBJECTS_WILL_BE_RELEASED_WHEN_QAPP_GOES_OUT_OF_SCOPE is implemented in both /Users/glados/opt/anaconda3/envs/pub-alfonso/lib/libQt6Core.6.10.2.dylib (0x15103a5b0) and /Users/glados/opt/anaconda3/envs/pub-alfonso/lib/libQt5Core.5.15.15.dylib (0x1454163d0). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[51433]: Class QT_ROOT_LEVEL_POOL__THESE_OBJECTS_WILL_BE_RELEASED_WHEN_QAPP_GOES_OUT_OF_SCOPE is implemented in both /Users/glados/opt/anaconda3/envs/pub-alfonso/li

Creating grids for PU model complete (region: global; duration: 0:02:52.867465)
Creating grids for PU model complete (region: North America; duration: 0:02:38.556302)
Creating grids for PU model complete (region: South America; duration: 0:02:45.082179)
Creating grids for PU model complete (region: Southeast Asia; duration: 0:02:39.402594)
Creating grids for PU model complete (region: Tethys; duration: 0:02:39.459439)
Creating grids for SVM model complete (region: global; duration: 0:02:36.822545)
Creating grids for SVM model complete (region: North America; duration: 0:02:35.692308)
Creating grids for SVM model complete (region: South America; duration: 0:02:37.213793)
